#TODO 
* Add a *previous_nerfed* and *previous_buff* flag to champions_pick_stats.
* The analytic table must have previous pick and ban rate too. And rate changes.
* Test the model with this few columns
    * logistic regresion and random forest with scikit-learn
* Add specific information about the champion (kda, kill streak, etc)

In [1]:
import pandas as pd
from transformations.utils import CHAMPIONS_CHANGES_GOLD, CHAMPIONS_PICK_STATS_FILE, PATCH_TOP_PICK_RATE_FILE

df = pd.read_parquet(CHAMPIONS_PICK_STATS_FILE)
df_top = pd.read_parquet(PATCH_TOP_PICK_RATE_FILE)

df.head(10)

,patch,champion,change_type,pro_pick_amt,pro_pick_rate,pro_ban_amt,pro_ban_rate,soloq_pick_amt,soloq_pick_rate,soloq_ban_amt,soloq_ban_rate
0,10.1,Aatrox,None,70.0,0.280,26.0,0.104,0.0,0.0,0.0,0.0
1,10.1,Ahri,None,1.0,0.004,0.0,0.000,0.0,0.0,0.0,0.0
2,10.1,Akali,None,19.0,0.076,221.0,0.884,0.0,0.0,0.0,0.0
3,10.1,Alistar,None,4.0,0.016,0.0,0.000,0.0,0.0,0.0,0.0
4,10.1,Amumu,None,0.0,0.000,1.0,0.004,0.0,0.0,0.0,0.0
5,10.1,Anivia,None,2.0,0.008,0.0,0.000,0.0,0.0,0.0,0.0
6,10.1,Annie,None,1.0,0.004,0.0,0.000,0.0,0.0,0.0,0.0
7,10.1,Aphelios,None,88.0,0.352,114.0,0.456,0.0,0.0,0.0,0.0
8,10.1,Ashe,None,4.0,0.016,0.0,0.000,0.0,0.0,0.0,0.0
9,10.1,Aurelion Sol,None,0.0,0.000,5.0,0.020,0.0,0.0,0.0,0.0


In [34]:
df.loc[df['patch'].str.startswith('2')]['patch'].unique()

<ArrowStringArray>
[   '25',  '25.1', '25.10', '25.11', '25.12', '25.13', '25.14', '25.15',
 '25.16', '25.17', '25.18', '25.19',  '25.2', '25.20', '25.21', '25.22',
 '25.23', '25.24',  '25.3',  '25.4',  '25.5',  '25.6',  '25.7',  '25.8',
  '25.9',  '26.1', '26.10', '26.11', '26.12', '26.13', '26.14', '26.15',
 '26.16',  '26.2',  '26.3',  '26.4',  '26.5',  '26.6',  '26.7',  '26.8',
  '26.9']
Length: 41, dtype: str

In [2]:


df_top_columns = df_top.columns.difference(['patch'])

df_with_top = (
    df
    .merge(df_top, on=['patch'], how='left')
    .assign(
        flg_top_pick_soloq_actual_patch = lambda x: x['soloq_top_pick_champion'] == x['champion'],
        flg_top_ban_soloq_actual_patch = lambda x: x['soloq_top_ban_champion'] == x['champion'],
        flg_top_pick_pro_actual_patch = lambda x: x['pro_top_pick_champion'] == x['champion'],
        flg_top_ban_pro_actual_patch = lambda x: x['pro_top_ban_champion'] == x['champion']
    )
    .drop(columns=df_top_columns, errors='ignore')
)

#df_with_top.head(20)

prev_cols = ['patch', 'change_type', 'pro_pick_amt', 'pro_pick_rate', 'pro_ban_amt', 'pro_ban_rate',
             'soloq_pick_amt', 'soloq_pick_rate', 'soloq_ban_amt', 'soloq_ban_rate',
             'flg_top_pick_soloq_actual_patch', 'flg_top_ban_soloq_actual_patch', 'flg_top_pick_pro_actual_patch', 'flg_top_ban_pro_actual_patch']


# Numeric patch ordering (string sort would put '25.10' before '25.2')
patch_key = df_with_top['patch'].astype(str).str.split('.', n=1, expand=True).fillna(0).astype(int)
df_sorted = (
    df_with_top
    .assign(_major=patch_key[0], _minor=patch_key[1])
    .sort_values(['champion', '_major', '_minor'])
)


# groupby keeps the shift inside each champion, so its first patch gets NaN instead of another champion's/last row
prev = df_sorted.groupby('champion')[prev_cols].shift(1).add_prefix('prev_')

df_final = (
    df_sorted
    .join(prev)
)

#df2.loc[df2['patch'] >= '25'][['champion', 'patch', 'change_type', 'prev_change']].sort_values(['champion', 'patch']).head(30)


In [ ]:
from transformations.utils import GOLD_DATASET_DIR

x = df_final.loc[df_final['champion'] == 'Jinx']
x.to_csv(f"{GOLD_DATASET_DIR}jinx_data.csv", index=False)

## 25.1 SHOW TWICE fixed
## Patch 3.15 (First patch i have information) show prev_patch as 26.9 (last one) fixed
## flg_top is not working fixed

In [3]:
from transformations.utils import CHAMPIONS_CHANGES_ANALYTICS_FILE
import pandas as pd

df_final.to_parquet(CHAMPIONS_CHANGES_ANALYTICS_FILE, index=False)

In [ ]:
#Now 